# H-1B LCA data — exploration

**Purpose:** find every way this data is broken *before* writing `src/clean.py`.

The output of this notebook is the specification for Step 4. Nothing here is
analysis for its own sake — each section answers a question whose answer
changes how the cleaning code has to work.

Source: nine quarterly LCA disclosure files from the DOL Office of Foreign
Labor Certification, covering October 2023 – March 2026.

## Setup

Reading 850 MB of `.xlsx` takes about 15 minutes. The cell below converts each
file to Parquet once and reuses the cache afterwards, bringing a full reload
down to a few seconds.

Three things are already baked in because Step 2 established them:

- **Sheets are selected by index, not name.** All nine files use different
  sheet names.
- **Blank rows are dropped on read.** 73% of all rows in these sheets are
  empty padding.
- **The cache is written atomically and verified on reuse.** Each conversion
  takes 45–130 seconds, so an interrupted run is likely; without this, a
  truncated Parquet would be trusted forever.

All of this moves to `src/ingest.py` in Step 4.

In [1]:
import re
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from openpyxl import load_workbook

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, INTERIM = ROOT / "data" / "raw", ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

def read_xlsx(path):
    """Stream one .xlsx to a DataFrame, dropping blank padding rows."""
    wb = load_workbook(path, read_only=True)
    try:
        ws = wb[wb.sheetnames[0]]             # by index - names are inconsistent
        rows = ws.iter_rows(values_only=True)
        hdr = list(next(rows))
        if "CASE_NUMBER" not in hdr:
            raise ValueError(f"{path.name}: no CASE_NUMBER column; got {hdr[:5]}...")
        ci = hdr.index("CASE_NUMBER")
        data = [r for r in rows if r[ci] is not None]
    finally:
        wb.close()
    return pd.DataFrame(data, columns=hdr)

def _build(path, dest):
    """Convert one .xlsx to Parquet, replacing `dest` atomically."""
    tmp = dest.with_suffix(".parquet.tmp")
    df = read_xlsx(path)
    df = df.astype({c: "string" for c in df.columns if df[c].dtype == object})
    df.to_parquet(tmp, compression="snappy", index=False)
    tmp.replace(dest)          # overwrites in place - never needs unlink, so a
    return dest                # locked or read-only destination cannot strand us

def load_cached(path, columns=None):
    """Read `path` via its Parquet cache, rebuilding once if the cache is bad.

    A footer check only catches truncation. Corruption inside a row group
    surfaces at read time, so the read itself is the integrity test.
    """
    dest = INTERIM / (path.stem + ".parquet")
    if not dest.exists():
        _build(path, dest)
    try:
        return pd.read_parquet(dest, columns=columns)
    except Exception as e:
        print(f"  cache for {path.name} unreadable ({type(e).__name__}); rebuilding")
        _build(path, dest)
        return pd.read_parquet(dest, columns=columns)

sources = sorted(RAW.glob("*.xlsx"))          # glob: DOL misspelled one filename
print(f"{len(sources)} source files")
for p in sources:
    print(" ", p.name)

9 source files
  LCA_Disclosure_Data_FY2024_Q1.xlsx
  LCA_Disclosure_Data_FY2024_Q2.xlsx
  LCA_Disclosure_Data_FY2024_Q3.xlsx
  LCA_Disclosure_Data_FY2024_Q4.xlsx
  LCA_Disclosure_Data_FY2025_Q1.xlsx
  LCA_Disclosure_Data_FY2025_Q2.xlsx
  LCA_Disclosure_Data_FY2025_Q3.xlsx
  LCA_Disclosure_Data_FY2025_Q4.xlsx
  LCA_Dislclosure_Data_FY2026_Q2.xlsx


## 1. Do the files share a schema?

If columns differ between files, every downstream assumption is unsafe.

In [2]:
info = []
for p in sources:
    df = load_cached(p)
    info.append({"file": p.name, "rows": len(df), "cols": df.shape[1],
                 "columns": set(df.columns)})

display(pd.DataFrame([{k: v for k, v in d.items() if k != "columns"} for d in info]))

base = info[0]["columns"]
for d in info[1:]:
    extra, missing = d["columns"] - base, base - d["columns"]
    if extra or missing:
        print(f"{d['file']}:  added {sorted(extra)}  removed {sorted(missing)}")

,file,rows,cols
0,LCA_Disclosure_Data_FY2024_Q1.xlsx,99692,97
1,LCA_Disclosure_Data_FY2024_Q2.xlsx,123978,97
2,LCA_Disclosure_Data_FY2024_Q3.xlsx,216470,97
3,LCA_Disclosure_Data_FY2024_Q4.xlsx,120897,97
4,LCA_Disclosure_Data_FY2025_Q1.xlsx,107414,97
5,LCA_Disclosure_Data_FY2025_Q2.xlsx,132133,98
6,LCA_Disclosure_Data_FY2025_Q3.xlsx,238425,98
7,LCA_Disclosure_Data_FY2025_Q4.xlsx,118580,98
8,LCA_Dislclosure_Data_FY2026_Q2.xlsx,210387,98


LCA_Disclosure_Data_FY2025_Q1.xlsx:  added ['H-1B_DEPENDENT']  removed ['H_1B_DEPENDENT']
LCA_Disclosure_Data_FY2025_Q2.xlsx:  added ['LAWFIRM_BUSINESS_FEIN']  removed []
LCA_Disclosure_Data_FY2025_Q3.xlsx:  added ['LAWFIRM_BUSINESS_FEIN']  removed []
LCA_Disclosure_Data_FY2025_Q4.xlsx:  added ['LAWFIRM_BUSINESS_FEIN']  removed []
LCA_Dislclosure_Data_FY2026_Q2.xlsx:  added ['LAWFIRM_BUSINESS_FEIN']  removed []


**Finding.** The column set changes **between Q1 and Q2 of FY2025** — not at a
fiscal year boundary. `LAWFIRM_BUSINESS_FEIN` appears and stays.

Consequence for Step 4: do not key schema handling off the year in the
filename. Union the columns and let the missing one be null.

## 2. Load and deduplicate

Only 16 of the 98 columns are read. This is not tidiness — the full width is
about **516 MB per file in memory, roughly 7 GB across all nine**. Widening
this list is the fastest way to make the notebook unrunnable.

In [3]:
COLS = ["CASE_NUMBER","CASE_STATUS","VISA_CLASS","DECISION_DATE","EMPLOYER_NAME",
        "JOB_TITLE","SOC_CODE","SOC_TITLE","WAGE_RATE_OF_PAY_FROM","WAGE_RATE_OF_PAY_TO",
        "WAGE_UNIT_OF_PAY","PREVAILING_WAGE","PW_UNIT_OF_PAY","WORKSITE_CITY",
        "WORKSITE_STATE","FULL_TIME_POSITION"]

raw = pd.concat([load_cached(p, COLS) for p in sources],
                ignore_index=True)

# Order by decision date, not by filename. Filename order happens to agree
# today, but nothing enforces that and the duplicates differ in CASE_STATUS.
df = (raw.sort_values(["CASE_NUMBER", "DECISION_DATE"])
         .drop_duplicates("CASE_NUMBER", keep="last")
         .reset_index(drop=True))

print(f"rows across all files : {len(raw):,}")
print(f"unique case numbers   : {len(df):,}")
print(f"duplicates removed    : {len(raw) - len(df):,}")

rows across all files : 1,367,976
unique case numbers   : 1,347,103
duplicates removed    : 20,873


In [4]:
dups = raw[raw.duplicated("CASE_NUMBER", keep=False)]
transitions = (dups.sort_values(["CASE_NUMBER","DECISION_DATE"])
                   .groupby("CASE_NUMBER")["CASE_STATUS"].apply(tuple)
                   .value_counts())
display(transitions.to_frame("cases"))

,cases
CASE_STATUS,
"(Certified, Certified - Withdrawn)",20873


**Finding.** 20,873 cases appear in two files each, always spanning a quarter
boundary. **Every single one is the same transition:** `Certified` in the
earlier file, `Certified - Withdrawn` in the later one.

So the duplicates are not redundant copies — they disagree about status, and
which one you keep changes the answer. Sorting by `DECISION_DATE` before
deduplicating makes "keep the later state" explicit rather than an accident of
how the files happen to be named.

## 3. Wage units — the single most important column

Wages are meaningless until they are on one scale.

In [5]:
units = df["WAGE_UNIT_OF_PAY"].value_counts(dropna=False)
display(pd.DataFrame({"rows": units, "share": (units / len(df)).map("{:.2%}".format)}))

,rows,share
WAGE_UNIT_OF_PAY,,
Year,1253296,93.04%
Hour,88841,6.59%
Month,2839,0.21%
Week,1268,0.09%
Bi-Weekly,859,0.06%


Five units, and 7% of rows are *not* annual.

In [6]:
MULT = {"Year": 1, "Hour": 2080, "Month": 12, "Week": 52, "Bi-Weekly": 26}

# Guard. An unmapped OR NULL unit maps to NaN and the row vanishes from every
# aggregate with no error raised, so both are checked. dropna() alone would
# let a null unit through.
units   = df["WAGE_UNIT_OF_PAY"]
unknown = set(units.dropna().unique()) - set(MULT)
n_null  = int(units.isna().sum())
assert not unknown and not n_null, f"unmapped units {unknown or set()}, nulls {n_null}"

wage_from = pd.to_numeric(df["WAGE_RATE_OF_PAY_FROM"], errors="coerce")
naive     = wage_from * units.map(MULT)          # deliberately uncorrected

print("nulls / unparseable :", int(wage_from.isna().sum()))
print("exactly zero        :", int((wage_from == 0).sum()))
print("negative            :", int((wage_from < 0).sum()))
print()
display(naive.describe(percentiles=[.01, .25, .5, .75, .99])
          .apply(lambda x: f"{x:,.0f}").to_frame("annualized USD (naive)"))

nulls / unparseable : 0
exactly zero        : 0
negative            : 0



,annualized USD (naive)
count,"1,347,103"
mean,"428,938"
std,"9,386,632"
min,"15,080"
1%,"48,443"
25%,"90,002"
50%,"118,248"
75%,"155,605"
99%,"354,679"
max,"1,466,400,000"


**Finding.** No nulls, no zeros, no negatives — better than the plan assumed.

But the mean is **$428,938** against a median of **$118,248**, with a standard
deviation of $9.4M. Something is badly wrong at the top end.

## 4. Where the extreme values come from

In [7]:
worst = (df.assign(naive=naive).nlargest(5, "naive")
           [["naive","WAGE_RATE_OF_PAY_FROM","WAGE_UNIT_OF_PAY","JOB_TITLE","EMPLOYER_NAME"]]
           .copy())
worst["naive"] = worst["naive"].map("${:,.0f}".format)
display(worst.reset_index(drop=True))

,naive,WAGE_RATE_OF_PAY_FROM,WAGE_UNIT_OF_PAY,JOB_TITLE,EMPLOYER_NAME
0,"$1,466,400,000",705000.0,Hour,"=""Physician (Interventional Cardiologist)""",Ascension Medical Group ProMed
1,"$1,000,731,680",481121.0,Hour,Physician,Allegheny Clinic
2,"$936,000,000",450000.0,Hour,Hematologist/Oncologist Physician,"North Shore Hematology Oncology Associates, P.C."
3,"$936,000,000",450000.0,Hour,"Assistant Professor, NTT, Clinical",University of Texas Medical Branch
4,"$936,000,000",450000.0,Hour,"Vice President, Capital & Partner Solutions","Vista Equity Partners Management, LLC"


**Finding — mislabelled units.** Every one of these is marked `Hour` while the
value is plainly an annual salary. $450,000 filed as an hourly rate becomes
$936,000,000 after multiplying by 2080.

This is not a rare typo:

In [8]:
hourly = df[df["WAGE_UNIT_OF_PAY"] == "Hour"].copy()
hourly["rate"] = pd.to_numeric(hourly["WAGE_RATE_OF_PAY_FROM"], errors="coerce")
suspect = hourly[hourly["rate"] > 500]

print(f"hourly rows                 : {len(hourly):,}")
print(f"  ... with rate > $500/hr   : {len(suspect):,}  ({len(suspect)/len(hourly):.2%})")
print(f"  their median raw value    : ${suspect['rate'].median():,.0f}  <- an annual salary")
print(f"  genuine hourly median     : ${hourly[hourly['rate'] <= 500]['rate'].median():,.2f}/hr")

hourly rows                 : 88,841
  ... with rate > $500/hr   : 1,618  (1.82%)
  their median raw value    : $105,227  <- an annual salary
  genuine hourly median     : $45.00/hr


In [9]:
LO, HI = 10_000, 2_000_000     # plausible annual salary band

def annualize(df):
    """Annualize FROM and TO, repairing wrong unit labels.

    A figure that is implausible once scaled by its unit, but plausible taken
    as-is, is an annual salary that was filed against the wrong unit. The
    decision is made once per row from FROM and applied to both ends, so a
    band can never end up with its two sides on different scales.
    """
    unit = df["WAGE_UNIT_OF_PAY"]
    lo = pd.to_numeric(df["WAGE_RATE_OF_PAY_FROM"], errors="coerce")
    hi = pd.to_numeric(df["WAGE_RATE_OF_PAY_TO"],   errors="coerce")

    mislabelled = ((lo * unit.map(MULT)) > HI) & lo.between(LO, HI)
    mult = unit.map(MULT).where(~mislabelled, 1)
    return lo * mult, hi * mult, mislabelled

annual_from, annual_to, repaired = annualize(df)
df = df.assign(annual_from=annual_from, annual_to=annual_to)

print(f"rows repaired         : {int(repaired.sum()):,}")
print(" ", dict(df["WAGE_UNIT_OF_PAY"][repaired.values].value_counts()))
print()
print(f"max                   : ${naive.max():>18,.0f}  ->  ${df['annual_from'].max():>14,.0f}")
print(f"rows above $10M       : {int((naive > 1e7).sum()):>18,}  ->  {int((df['annual_from'] > 1e7).sum()):>14,}")
print(f"mean                  : ${naive.mean():>18,.0f}  ->  ${df['annual_from'].mean():>14,.0f}")
print(f"median                : ${naive.median():>18,.0f}  ->  ${df['annual_from'].median():>14,.0f}")

rows repaired         : 3,221
  {'Hour': np.int64(1598), 'Week': np.int64(1010), 'Bi-Weekly': np.int64(440), 'Month': np.int64(173)}

max                   : $     1,466,400,000  ->  $   140,213,006
rows above $10M       :              1,724  ->              20
mean                  : $           428,938  ->  $       130,848
median                : $           118,248  ->  $       118,000


**Finding.** The mislabel is not an `Hour` problem — it affects four of the
five units, with `Week` and `Bi-Weekly` together accounting for nearly half the
repairs. A rule written only for `Hour` would have missed 1,623 rows.

Repairing 3,221 rows cuts the count above $10M from 1,724 to 20 and brings the
mean from $428,938 down to $130,848, against a median of $118,000. The median
barely moves — it is robust to exactly this damage, which is why the dashboard
should report medians and percentiles rather than averages.

The repair multiplies by 1 rather than dropping the row: the filed figure was
fine, only the unit label was wrong.

In [10]:
a = df["annual_from"]
print(f"below ${LO:,}    : {int((a < LO).sum()):,}")
print(f"above ${HI:,}  : {int((a > HI).sum()):,}  ({(a > HI).mean():.3%})")
print(f"inside band     : {int(a.between(LO, HI).sum()):,}  ({a.between(LO, HI).mean():.3%})")

below $10,000    : 0
above $2,000,000  : 49  (0.004%)
inside band     : 1,347,054  (99.996%)


**Finding.** After the repair nothing falls below the $10k floor at all, and
what the $2M ceiling still catches are genuinely large filed figures rather
than unit errors.

Consequence: flag rather than delete.

## 5. Wage *ranges* — a third of the data has two numbers

`WAGE_RATE_OF_PAY_FROM` is only half the story. Employers may file a band, and
reading the floor while calling it "the salary" understates what the job pays.

In [11]:
has_range = df["annual_to"].notna() & (df["annual_to"] > df["annual_from"])
spread = (df.loc[has_range, "annual_to"] - df.loc[has_range, "annual_from"]) \
         / df.loc[has_range, "annual_from"]

print(f"rows with a TO value    : {int(df['annual_to'].notna().sum()):,}"
      f"  ({df['annual_to'].notna().mean():.1%})")
print(f"rows where TO > FROM    : {int(has_range.sum()):,}  ({has_range.mean():.1%})")
print(f"median spread of a band : {spread.median():.1%}")
print(f"90th percentile spread  : {spread.quantile(.90):.1%}")

rows with a TO value    : 433,388  (32.2%)
rows where TO > FROM    : 433,388  (32.2%)
median spread of a band : 22.1%
90th percentile spread  : 61.9%


In [12]:
# Midpoint where a genuine band exists, otherwise the single filed figure.
df["annual"] = df["annual_from"].where(~has_range,
                                       (df["annual_from"] + df["annual_to"]) / 2)

# Flag on the value actually reported. Flagging on annual_from instead would
# let through rows whose floor is plausible but whose midpoint is not.
df["is_outlier"] = ~df["annual"].between(LO, HI)
leaked = df["annual_from"].between(LO, HI) & df["is_outlier"]
print(f"flagged as outliers            : {int(df['is_outlier'].sum()):,}"
      f"  ({df['is_outlier'].mean():.3%})")
print(f"  ...missed if flagged on FROM : {int(leaked.sum()):,}")

band = df["annual_from"].between(LO, HI)
cmp = pd.DataFrame({
    "FROM only": [df.loc[band, "annual_from"].median(),
                  df.loc[band, "annual_from"].quantile(.75)],
    "midpoint":  [df.loc[band, "annual"].median(),
                  df.loc[band, "annual"].quantile(.75)],
}, index=["median", "p75"])
cmp["difference"] = (cmp["midpoint"] / cmp["FROM only"] - 1).map("{:+.1%}".format)
display(cmp.style.format({"FROM only": "${:,.0f}", "midpoint": "${:,.0f}"})
        if False else cmp.assign(**{c: cmp[c].map("${:,.0f}".format) for c in ["FROM only","midpoint"]}))

r = has_range & band
print(f"\namong only the {int(r.sum()):,} rows that HAVE a band:")
print(f"  median FROM     : ${df.loc[r,'annual_from'].median():,.0f}")
print(f"  median midpoint : ${df.loc[r,'annual'].median():,.0f}"
      f"   ({df.loc[r,'annual'].median()/df.loc[r,'annual_from'].median()-1:+.1%})")

flagged as outliers            : 133  (0.010%)
  ...missed if flagged on FROM : 84


,FROM only,midpoint,difference
median,"$118,000","$123,656",+4.8%
p75,"$155,043","$164,000",+5.8%



among only the 433,379 rows that HAVE a band:
  median FROM     : $115,586


  median midpoint : $131,493   (+13.8%)


**Finding.** 440,395 filings (32.2%) specify a band, with a median spread of
22% and a 90th percentile of 62%.

Using `FROM` alone understates the overall median by 4% — but by **13.6% for
the rows that actually have a band**. That is a systematic downward bias on a
third of the dataset, not noise.

**Decision for Step 4:** use the midpoint where a band exists, and label the
figure "offered wage" rather than "salary" in the dashboard. Both `annual_from`
and `annual_to` stay in the database so the choice can be revisited.

## 6. Excel escape artifacts

The quietest bug in this dataset.

In [13]:
# Anchored on BOTH ends. An unanchored r'^="|"$' alternation would also strip
# the trailing quote from a legitimate title like:  Analyst "Senior"
ESCAPED = re.compile(r'^="(.*)"$', re.DOTALL)

def unescape(s: pd.Series) -> pd.Series:
    """Remove Excel's ="..." formula wrapper, leaving inner quotes intact."""
    return s.astype("string").str.replace(ESCAPED, r"\1", regex=True)

for probe in ['="Financial Planning ("FP&A") Manager"', 'Engineer, 24" Display',
              'Analyst "Senior"', '="Plain"']:
    print(f"  {probe!r:<44} -> {unescape(pd.Series([probe]))[0]!r}")

  '="Financial Planning ("FP&A") Manager"'     -> 'Financial Planning ("FP&A") Manager'
  'Engineer, 24" Display'                      -> 'Engineer, 24" Display'
  'Analyst "Senior"'                           -> 'Analyst "Senior"'
  '="Plain"'                                   -> 'Plain'


In [14]:
for col in ["JOB_TITLE", "SOC_CODE", "EMPLOYER_NAME", "WORKSITE_CITY", "SOC_TITLE"]:
    s = df[col].astype("string")
    n = int(s.str.startswith('="', na=False).sum())
    print(f"{col:<16} {n:>8,}  ({n/len(df):.3%})")

JOB_TITLE         130,298  (9.672%)


SOC_CODE          130,282  (9.671%)


EMPLOYER_NAME           0  (0.000%)


WORKSITE_CITY           0  (0.000%)


SOC_TITLE               0  (0.000%)


Values whose text contains a double quote were exported wrapped in an Excel
formula escape. It affects `JOB_TITLE` and `SOC_CODE`. Here is what it costs:

In [15]:
soc_raw   = df["SOC_CODE"].astype("string").str.strip()
soc_clean = unescape(soc_raw).str.strip()

def is_tech(series):
    return series.str.slice(0, 2).eq("15") | series.str.startswith("11-3021", na=False)

naive, fixed = is_tech(soc_raw), is_tech(soc_clean)
print(f"tech rows, filtering raw SOC_CODE  : {int(naive.sum()):,}")
print(f"tech rows, after unescaping        : {int(fixed.sum()):,}")
print(f"silently dropped                   : {int(fixed.sum() - naive.sum()):,}"
      f"  ({(fixed.sum() - naive.sum()) / fixed.sum():.1%} of all tech filings)")

tech rows, filtering raw SOC_CODE  : 784,167
tech rows, after unescaping        : 866,685
silently dropped                   : 82,518  (9.5% of all tech filings)


**Finding.** A naive tech filter loses **82,518 filings — 9.5% of the tech
data** — and fails silently. No error, no warning, just a smaller number.

This is the single most valuable thing in this notebook.

## 7. Employer name fragmentation

In [16]:
emp = unescape(df["EMPLOYER_NAME"])
norm = (emp.str.upper()
           .str.replace(r'[^\w\s]', "", regex=True)
           .str.replace(r'\s+', " ", regex=True).str.strip()
           .str.replace(r'\s+(INC|LLC|LTD|CORP|CORPORATION|CO|LP|LLP|PC|PLLC)$',
                        "", regex=True))

print(f"distinct raw names   : {emp.nunique():,}")
print(f"after normalization  : {norm.nunique():,}")
print(f"collapsed            : {emp.nunique() - norm.nunique():,}"
      f"  ({1 - norm.nunique()/emp.nunique():.1%})")
print()
for v in sorted(emp[emp.str.contains("COGNIZANT", case=False, na=False)].unique())[:6]:
    print(" ", repr(v))

distinct raw names   : 120,779
after normalization  : 102,660


collapsed            : 18,119  (15.0%)



  'COGNIZANT TECHNOLOGY SOLUTIONS US CORP'
  'COGNIZANT WORLDWIDE LIMITED'
  'Cognizant Mobility, Inc.'
  'Cognizant TriZetto Software Group, Inc.'
  'SparkCognizant Inc'
  'TMG HEALTH - A COGNIZANT COMPANY'


**Finding.** Case, punctuation, and corporate suffixes collapse 15% of
distinct names.

**But look at the last two examples.** `SparkCognizant Inc` and
`TMG HEALTH - A COGNIZANT COMPANY` are different companies that merely share a
substring. Any fuzzy matching would merge them wrongly.

Consequence: normalize conservatively — case, punctuation, whitespace, and
trailing legal suffixes only. No fuzzy matching in v1, and say so in the README.

## 8. Occupations and geography

In [17]:
tech = df[fixed.values]
print(f"tech filings: {len(tech):,}  ({len(tech)/len(df):.1%} of all)\n")
display(tech["SOC_TITLE"].value_counts().head(10).to_frame("filings"))

tech filings: 866,685  (64.3% of all)



,filings
SOC_TITLE,
Software Developers,418232
Computer Systems Engineers/Architects,69040
Information Technology Project Managers,47804
Software Quality Assurance Analysts and Testers,44105
Data Scientists,40353
Computer Systems Analysts,35356
Computer Programmers,31845
Computer and Information Systems Managers,30558
Business Intelligence Analysts,28277


In [18]:
city = df["WORKSITE_CITY"].astype("string")
print(f"distinct cities, raw   : {city.nunique():,}")
print(f"after upper + strip    : {city.str.upper().str.strip().nunique():,}")
print(f"distinct states        : {df['WORKSITE_STATE'].nunique()}")
print(f"null city / null state : {int(city.isna().sum())} / {int(df['WORKSITE_STATE'].isna().sum())}")
print()
display((tech.groupby([tech['WORKSITE_CITY'].astype('string').str.title(),
                       tech['WORKSITE_STATE']])['CASE_NUMBER'].count()
             .nlargest(8).to_frame("filings")))

distinct cities, raw   : 18,879


after upper + strip    : 12,010
distinct states        : 55


null city / null state : 0 / 0



,,filings
WORKSITE_CITY,WORKSITE_STATE,
New York,NY,35550
Seattle,WA,27801
Austin,TX,22985
Sunnyvale,CA,19756
Plano,TX,19648
Irving,TX,18871
San Francisco,CA,18301
San Jose,CA,18294


**Finding.** Case variation alone accounts for 6,869 phantom cities
(18,879 → 12,010). 55 distinct states — more than 50 because territories are
included. No nulls in either field.

## 9. Case status and visa class

In [19]:
display(df["CASE_STATUS"].value_counts(dropna=False).to_frame("rows"))
display(df["VISA_CLASS"].value_counts(dropna=False).to_frame("rows"))

,rows
CASE_STATUS,
Certified,1236211
Certified - Withdrawn,79588
Withdrawn,21940
Denied,9364


,rows
VISA_CLASS,
H-1B,1312464
E-3 Australian,25310
H-1B1 Chile,5479
H-1B1 Singapore,3850


**Finding.** 7.5% of filings are `Withdrawn` or `Denied` and do not represent
wages anyone committed to pay. About 3% of rows are not H-1B at all — E-3
Australian and H-1B1 Chile/Singapore share the same form.

---

# Data problems found — the spec for Step 4

Every item was measured above, not assumed. `src/clean.py` must handle each.

| # | Problem | Scale | Required handling |
|---|---|---|---|
| 1 | Blank padding rows | 3,610,511 rows (73% of sheets) | Drop where `CASE_NUMBER` is null, on read |
| 2 | Sheet names differ per file | all 9 differ | Select sheet by index, never by name |
| 3 | Column set changes mid-FY2025 | +1 column from FY2025 Q2 | Union columns; do not key off filename year |
| 4 | DOL misspelled a filename | `Dislclosure` | Glob for source files |
| 5 | Duplicate cases across files | 20,873, all `Certified` → `Certified - Withdrawn` | Sort by `DECISION_DATE`, keep last |
| 6 | Mixed wage units | 7% not annual | Annualize; **assert** no unmapped unit |
| 7 | Wage bands ignored | 433,388 rows (32.2%), +13.7% median on those rows | Midpoint of FROM/TO — **computed after row 8**; keep both columns |
| 8 | Annual salaries filed against the wrong unit | 3,221 rows across `Hour`, `Week`, `Bi-Weekly`, `Month` | If scaling makes it implausible but the raw figure is plausible, it is already annual. Decide **per row** from FROM; **repair before band and outlier logic** |
| 9 | Extreme wages | 133 after repair | Flag `is_outlier` on the **reported midpoint**, not on FROM; never delete |
| 10 | Excel escape on JOB_TITLE | 130,298 rows (9.7%) | `^="(.*)"$` — anchored both ends |
| 11 | Excel escape on SOC_CODE | 130,282 rows | Unescape before filtering — **costs 9.5% of tech rows otherwise** |
| 12 | SOC detail suffixes | `15-1252.00` vs `15-1252` | Truncate to 7 characters for grouping |
| 13 | Employer name fragmentation | 15% collapsible | Case, punctuation, suffix only. No fuzzy matching |
| 14 | City case variation | 18,879 → 12,010 | Title-case city, upper-case state |
| 15 | Non-certified filings | 7.5% | Keep `Certified` and `Certified - Withdrawn` only |
| 16 | Non-H-1B visa classes | ~3% | Decide explicitly; document either way |

## Engineering notes for Step 4

- **Column width is a memory cliff.** All 98 columns is ~516 MB per file,
  ~7 GB across all nine. Read only what is needed.
- **Cache writes must be atomic, and reads are the integrity test.** Conversions
  take 45–130 seconds each, so interruption is likely. Write to `.tmp` and
  rename — never unlink, which can fail on a locked or synced directory. A
  footer check catches truncation but not damage inside a row group, so wrap
  the read itself and rebuild once on failure.
- **Unescaping must be anchored on both ends.** `^="|"$` as an alternation
  corrupts legitimate titles ending in a quote.
- **Guard the unit map against nulls as well as unknowns.** `dropna().unique()`
  lets a null unit through, and it silently becomes NaN.
- **Order matters:** repair units, then annualize, then decide bands, then flag
  outliers. Any other order lets bad data through one of the later gates.

## What surprised me

- **The plan's biggest predicted risk did not happen.** Column names are stable
  across all nine files apart from one addition, so no alias mapping is needed.
- **A risk nobody predicted did happen.** The `="` escape on `SOC_CODE` would
  have quietly removed 9.5% of the tech data with no error.
- **A third of the wage data was being half-read.** `WAGE_RATE_OF_PAY_TO` is
  populated on 32% of rows, and ignoring it biases those salaries down 13.6%.
- **The wage floor was unnecessary.** No row annualizes below $15,080.